# Wild-Diff-ICMH — Phase 2 (2/2): baseline và MegaDetector trước/sau decode

Mục tiêu: (1) các baseline nén để so với B0, (2) MegaDetector trên ảnh **trước** (ảnh gốc) và **sau** khi
decode để so bbox vùng động vật — mAP, mất con vật, con vật "ảo". Mọi phương pháp được chấm trên **đúng
cùng các ảnh B0 đã decode** (100 ảnh dev, cạnh 512), nên so sánh là so cặp từng ảnh.

Notebook `Wild_Diff_ICMH_Phase2_Prepare.ipynb` (tập dev, MegaDetector trên ảnh gốc, decode B0) đã chạy xong.

| Bước | Việc | Thời gian |
|---|---|---|
| 1–3 | Drive, code nhánh `phase2`, cài đặt | ~10 phút |
| 4 | Chép ảnh dev, kiểm tra kho B0, dựng MegaDetector | ~5 phút |
| 5 | JPEG/WebP: nén rồi chấm từng điểm (15 điểm) | ~20 phút |
| 6 | CompressAI: nén rồi chấm từng điểm (18 điểm) | ~30 phút |
| 7 | Chấm 6 kho B0 | ~20 phút |
| 8 | Đồ thị RD | vài giây |
| 9 | Bảng kết quả để gửi lại | vài giây |
| 10 | Đẩy dữ liệu lên Drive, **tự ngắt runtime** | vài giây |

Bấm **Runtime → Run all** rồi để máy chạy; Bước 10 tự ngắt runtime khi xong. Mỗi điểm được chấm và ghi
lên Drive **ngay sau khi nén xong**, nên runtime bị ngắt giữa chừng chỉ mất điểm đang làm dở: mở lại
notebook, chạy lại **từ Bước 1** — điểm đã chấm tự bỏ qua. Kết quả: `MyDrive/wild_diff_icmh/phase2/`
và `results/results.jsonl`.

## Bước 1 — Gắn Google Drive (mỗi runtime mới)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Bước 2 — Lấy mã nguồn và kiểm tra GPU (mỗi runtime mới)

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/TranDuon/Wild-Diff-ICMH.git'
BRANCH = 'phase2'  # Phase 2 branch; set back to 'main' when merging
REPO = Path('/content/Wild-Diff-ICMH')

if (REPO / '.git').is_dir():
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run(['nvidia-smi'], check=True)
print('Mã nguồn:', REPO)


## Bước 3 — Cài dependencies an toàn (mỗi runtime mới)

Cell này chủ động giữ nguyên NumPy/SciPy/Torch có sẵn của Colab,
cài đủ dependency runtime của CompressAI, rồi import `TagGCM` để bắt lỗi ngay tại đây.

In [ ]:
source_requirements = REPO / 'requirements-colab.txt'
safe_requirements = Path('/content/requirements-colab-safe.txt')
core_constraints = Path('/content/colab-core-constraints.txt')

from importlib import metadata as importlib_metadata

core_distributions = ('numpy', 'scipy', 'torch', 'torchvision')
core_versions_before = {
    name: importlib_metadata.version(name)
    for name in core_distributions
}

filtered_lines = []
for line in source_requirements.read_text(encoding='utf-8').splitlines():
    normalized = line.strip().lower()
    if normalized.startswith(('numpy', 'scipy')):
        print('Giữ bản Colab, bỏ qua:', line)
        continue
    filtered_lines.append(line)

safe_requirements.write_text(
    '\n'.join(filtered_lines) + '\n',
    encoding='utf-8',
)

core_constraints.write_text(
    '\n'.join(
        f'{name}=={version}'
        for name, version in core_versions_before.items()
    ) + '\n',
    encoding='utf-8',
)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(safe_requirements),
    '-c', str(core_constraints),
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-deps', '--no-build-isolation',
    'compressai==1.2.8',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    '--force-reinstall', 'src/recognize-anything',
], cwd=REPO, check=True)

import importlib
import numpy as np
import numpy.random as npr
import scipy
import torch
import torch_geometric
import pytorch_msssim
import compressai.ans
import compressai.entropy_models
import compressai.losses
import compressai.zoo

# A pip subprocess can add files after this Jupyter kernel initialized.
# Refresh import caches so the freshly installed vendored package is
# visible immediately, without requiring a runtime restart.
importlib.invalidate_caches()
import ram
from model.lfgcm import TagGCM

core_versions_after = {
    name: importlib_metadata.version(name)
    for name in core_distributions
}
assert core_versions_after == core_versions_before, (
    'Dependency install changed Colab core packages: '
    f'before={core_versions_before}, after={core_versions_after}'
)
assert torch.cuda.is_available(), 'Vào Runtime > Change runtime type > chọn GPU'
gpu = torch.cuda.get_device_properties(0)
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, np.__file__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__, torch.version.cuda)
print('PyG:', torch_geometric.__version__)
print('RAM package:', ram.__file__)
print('Core packages preserved:', core_versions_after)
print('GPU:', gpu.name, f'{gpu.total_memory / 2**30:.1f} GiB')
print('Kiểm tra numpy.random:', npr.rand(3))
print('THÀNH CÔNG: môi trường và TagGCM đã sẵn sàng.')


## Bước 4 — Chuẩn bị (ảnh dev, đường dẫn, kiểm tra B0, MegaDetector)

Chỉ chép **300 ảnh dev** (~250 MB) từ Drive — không cần 10.222 ảnh hay checkpoint. Kiểm tra các kết quả của
notebook Prepare (nhãn ngày/đêm, tập dev, MegaDetector trên ảnh gốc, kho B0) và in số ảnh B0 đã decode.
Chọn `BASELINE_DEV` = đúng tập ảnh B0 đã decode ở cạnh 512 (100 ảnh) để baseline và B0 so cặp trên cùng
ảnh. Dựng môi trường MegaDetector riêng và hàm `score_archive` chấm một kho (dùng ở Bước 5, 6, 7).

In [ ]:
import json
import shutil
from concurrent.futures import ThreadPoolExecutor

import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/wild_diff_icmh')
LOCAL_IMAGES = Path('/content/data/wild_diff_icmh/images')
P2 = DRIVE_ROOT / 'phase2'
MANIFEST = 'data/manifests/kgalagadi_site_split.jsonl'
ILLUMINATION = P2 / 'kgalagadi_illumination.jsonl'
DEV_LIST = P2 / 'kgalagadi_dev.txt'
DETECTIONS = P2 / 'detections' / 'originals.jsonl'
ARCHIVE = P2 / 'archive'
BASELINE_ROOT = Path('/content/p2_baselines')
BITSTREAMS = P2 / 'bitstreams'
EVAL_DIR = P2 / 'eval'
RESULTS_REGISTRY = DRIVE_ROOT / 'results' / 'results.jsonl'
for folder in (EVAL_DIR, BITSTREAMS, BASELINE_ROOT, DRIVE_ROOT / 'logs'):
    folder.mkdir(parents=True, exist_ok=True)
for needed, step in ((ILLUMINATION, 'P2-1'), (DEV_LIST, 'P2-2'), (DETECTIONS, 'P2-3')):
    assert needed.is_file(), f'Notebook Prepare chưa xong {step}: chưa có {needed}'

def run_logged(command, log_name, echo=True):
    log_path = DRIVE_ROOT / 'logs' / log_name
    command = [str(part) for part in command]
    print('$', ' '.join(command))
    with log_path.open('a', encoding='utf-8') as log_stream:
        process = subprocess.Popen(
            command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            if echo:
                print(line, end='')
            log_stream.write(line)
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'{log_name}: dừng với mã {return_code}; log đầy đủ: {log_path}')


def dev_ids(path):
    return [line for line in Path(path).read_text().splitlines() if line and not line.startswith('#')]


def scored(curve, point):
    return (EVAL_DIR / f'{curve}__{point}.done.json').is_file()


wanted = set(dev_ids(DEV_LIST))
rows = [json.loads(line) for line in (REPO / MANIFEST).read_text(encoding='utf-8').splitlines() if line.strip()]
to_copy = [row['relative_path'] for row in rows
           if row['image_id'] in wanted and not (LOCAL_IMAGES / row['relative_path']).is_file()]

def copy_dev_image(relative):
    target = LOCAL_IMAGES / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DRIVE_ROOT / 'images' / relative, target)

with ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(copy_dev_image, to_copy))
print(f'Ảnh dev cục bộ: {len(wanted)} (vừa chép {len(to_copy)})')

print('Kho B0 (ảnh đã decode / ảnh yêu cầu):')
b0_runs = sorted(ARCHIVE.glob('B0_*/*/run_info.json'))
assert b0_runs, 'Chưa có kho B0 nào trong phase2/archive (notebook Prepare, P2-7)'
for run_info in b0_runs:
    requested = dev_ids(json.loads(run_info.read_text()).get('dev_list') or DEV_LIST)
    logged = {json.loads(line)['image_id'] for line in (run_info.parent / 'decode_log.jsonl').read_text().splitlines() if line.strip()}
    done = len(set(requested) & logged)
    print(f'  {run_info.parent.parent.name}/{run_info.parent.name}: {done}/{len(requested)}'
          + ('' if done == len(requested) else '  ← CHƯA ĐỦ'))

# Baselines are coded and scored on exactly the largest image set B0 decoded, so every
# comparison (image metrics and MegaDetector before/after) is paired image by image.
BASELINE_DEV = max((Path(json.loads(run_info.read_text()).get('dev_list') or DEV_LIST) for run_info in b0_runs),
                   key=lambda path: len(dev_ids(path)))
print(f'Baseline chạy trên {BASELINE_DEV.name}: {len(dev_ids(BASELINE_DEV))} ảnh (cùng ảnh với B0)')

# Colab's Python has no ensurepip, so ``python -m venv`` cannot install pip;
# virtualenv ships its own.  --system-site-packages reuses Colab's torch.
DETECT_ENV = Path('/content/envs/detect')
DETECT_PY = DETECT_ENV / 'bin' / 'python'
DETECT_READY = DETECT_ENV / '.ready'
if not DETECT_READY.exists():
    shutil.rmtree(DETECT_ENV, ignore_errors=True)  # leftovers of a failed attempt
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], 'p2_detect_env.log')
    run_logged([sys.executable, '-m', 'virtualenv', '--system-site-packages', DETECT_ENV], 'p2_detect_env.log')
    # The env also sees Colab's transformers, which needs Colab's
    # huggingface-hub (<1.0); PytorchWildlife's deps would otherwise pull 1.x.
    import importlib.metadata
    hub_pin = f"huggingface-hub=={importlib.metadata.version('huggingface_hub')}"
    run_logged([DETECT_PY, '-m', 'pip', 'install', '-q', 'PytorchWildlife', hub_pin], 'p2_detect_env.log')
    run_logged([DETECT_PY, '-c', 'from PytorchWildlife.models import detection; print("PytorchWildlife OK")'],
               'p2_detect_env.log')
    DETECT_READY.touch()

try:
    import pycocotools  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pycocotools'], check=True)

import tarfile


def pack_bitstreams(root, curve, point):
    # One file on Drive per point instead of hundreds of small writes.
    target = BITSTREAMS / curve / f'{point}.tar'
    target.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(target, 'w') as bundle:
        for path in sorted(root.rglob('*')):
            if path.is_file() and (path.parent.name == 'data' or path.name in ('decode_log.jsonl', 'run_info.json')):
                bundle.add(path, arcname=path.relative_to(root).as_posix())


def score_archive(root):
    # Image metrics + MegaDetector for one archive <curve>/<point>; results reach Drive at once.
    root = Path(root)
    run_info = root / 'run_info.json'
    if not run_info.is_file():
        return  # baseline point already scored (local folder removed) or never coded
    curve, point = root.parent.name, root.name
    name = f'{curve}__{point}'
    image_eval = EVAL_DIR / f'{name}.jsonl'
    machine_eval = EVAL_DIR / f'{name}.machine.json'
    archived = sum(1 for line in (root / 'decode_log.jsonl').read_text().splitlines() if line.strip())
    done_marker = EVAL_DIR / f'{name}.done.json'
    if done_marker.is_file() and json.loads(done_marker.read_text())['images'] >= archived:
        print('Đã chấm:', name, f'({archived} ảnh)')
        return
    info = json.loads(run_info.read_text())
    # Score the image set this archive was asked for (a B0 subset), never
    # leftovers of an earlier, differently sized run in the same folder.
    archive_dev = info.get('dev_list') or DEV_LIST
    common = ['--manifest', MANIFEST, '--split', 'val', '--dev-list', archive_dev,
              '--illumination-sidecar', ILLUMINATION]
    run_logged([
        sys.executable, '-u', 'tools/evaluate_kgalagadi.py', *common,
        '--data-root', LOCAL_IMAGES, '--reconstruction-root', root,
        '--detections', DETECTIONS, '--method', curve, '--lpips', '--dists', '--archived-only',
        '--output', image_eval, '--results-registry', RESULTS_REGISTRY,
        '--exp-id', f'p2dev_{name}', '--lambda-rate', point,
        '--ddim-steps', info.get('steps', 0),
    ], f'p2_eval_{name}.log', echo=False)
    predictions = DETECTIONS.parent / f'{name}.jsonl'
    run_logged([
        DETECT_PY, '-u', 'tools/detect/run_megadetector.py', '--manifest', MANIFEST,
        '--split', 'val', '--dev-list', archive_dev, '--image-root', root, '--suffix', '.png', '--archived-only',
        '--output', predictions,
    ], f'p2_megadetector_{name}.log', echo=False)
    run_logged([
        sys.executable, '-u', 'tools/eval_machine.py', *common,
        '--gt', DETECTIONS, '--pred', predictions, '--output', machine_eval, '--archived-only',
        '--results-registry', RESULTS_REGISTRY, '--exp-id', f'p2dev_{name}',
        '--method', curve, '--lambda-rate', point, '--ddim-steps', info.get('steps', 0),
    ], f'p2_machine_{name}.log', echo=False)
    baseline = BASELINE_ROOT in root.parents
    if baseline:
        pack_bitstreams(root, curve, point)  # before the marker: a crash here re-scores, never loses the tar
    done_marker.write_text(json.dumps({'images': archived}))
    if baseline:
        shutil.rmtree(root)  # ~3 GB of PNGs per point; reproducible from the bitstream tar
    print('Đã chấm:', name, f'({archived} ảnh)')


print('Sẵn sàng.')


## Bước 5 — Baseline JPEG / WebP (CPU, chạy song song 4 tiến trình)

JPEG ở độ phân giải gốc, ở cạnh dài 1024 và 512; WebP ở 1024 và 512. Cùng giao thức: ảnh gốc vào, ảnh gốc ra.

Ảnh tái tạo của baseline (~10 MB/ảnh PNG) ghi vào ổ cục bộ `/content/p2_baselines`, **không ghi lên
Drive** — ghi hàng chục GB lên Drive làm Colab bị chặn ("Google Drive quota exceeded"). Chúng tính lại
được từ bitstream trong vài giây. Mỗi điểm được **chấm ngay khi nén xong** (`score_archive`): kết quả
lên Drive, bitstream gói thành một file `.tar` lên Drive, ảnh cục bộ bị xoá. Điểm nào đã chấm thì bỏ qua
cả bước nén.

In [ ]:
CLASSICAL = (
    [('jpeg', q, None) for q in (5, 15, 40)]
    + [('jpeg', q, side) for q in (5, 15, 40) for side in (1024, 512)]
    + [('webp', q, side) for q in (5, 15, 40) for side in (1024, 512)]
)

def run_classical(job):
    codec, quality, side = job
    curve = f"{codec}_{'full' if side is None else f'ls{side}'}"
    if scored(curve, f'q{quality}'):
        return curve, quality
    command = [
        sys.executable, '-u', 'tools/baselines/run_classical.py', '--manifest', MANIFEST,
        '--data-root', LOCAL_IMAGES, '--split', 'val', '--dev-list', BASELINE_DEV,
        '--codec', codec, '--quality', quality, '--output', BASELINE_ROOT / curve / f'q{quality}',
        '--skip-existing',
    ]
    if side:
        command += ['--processing-long-side', side]
    run_logged(command, f'p2_{curve}_q{quality}.log', echo=False)
    return curve, quality

with ThreadPoolExecutor(max_workers=4) as pool:
    for curve, quality in pool.map(run_classical, CLASSICAL):
        score_archive(BASELINE_ROOT / curve / f'q{quality}')


## Bước 6 — Baseline CompressAI (GPU, nhẹ)

`bmshj2018-hyperprior` (họ codec của bài Xie 2025), `mbt2018`, `cheng2020-attn`, chất lượng 1–3,
ở cạnh dài 1024 và 512.

- `bmshj2018-hyperprior`: entropy coding thật, bitstream ghi ra file.
- `mbt2018`, `cheng2020-attn` là model **tự hồi quy**: entropy coder của chúng chạy từng vị trí latent
  một (~29 s/ảnh ở 1024 — lần chạy 05/10 mất ~2,4 giờ cho một điểm 300 ảnh). Ở đây chúng chạy một lượt forward và bpp = tổng −log2 likelihood (chế độ
  `--entropy-estimation` của CompressAI), ghi `rate: estimated` trong `run_info.json` và registry.
  Chênh lệch với bitstream thật chỉ là vài byte header mỗi ảnh.

Mỗi điểm được chấm ngay khi nén xong.

In [ ]:
for model in ('bmshj2018-hyperprior', 'mbt2018', 'cheng2020-attn'):
    for quality, side in [(q, s) for q in (1, 2, 3) for s in (1024, 512)]:
        curve = f'compressai-{model}_ls{side}'
        if scored(curve, f'q{quality}'):
            continue
        run_logged([
            sys.executable, '-u', 'tools/baselines/run_compressai_zoo.py', '--manifest', MANIFEST,
            '--data-root', LOCAL_IMAGES, '--split', 'val', '--dev-list', BASELINE_DEV,
            '--model', model, '--quality', quality, '--processing-long-side', side,
            '--output', BASELINE_ROOT / curve / f'q{quality}', '--skip-existing',
        ], f'p2_{curve}_q{quality}.log', echo=False)
        score_archive(BASELINE_ROOT / curve / f'q{quality}')


## Bước 7 — Chấm các kho B0 (và điểm baseline còn sót)

Mỗi kho B0 trên Drive (`phase2/archive/B0_*/<điểm>/`): chỉ số ảnh (PSNR, MS-SSIM, LPIPS, DISTS, bpp…) và
MegaDetector trên ảnh decode, so với MegaDetector trên ảnh gốc → mAP, AP_small, mất con vật, con vật
"ảo", ảnh rỗng báo nhầm. Bbox dự đoán trên ảnh decode lưu ở `phase2/detections/<đường>__<điểm>.jsonl`.
Kết quả vào `results/results.jsonl` với `exp_id = p2dev_<đường>__<điểm>`. Kho đã chấm thì bỏ qua.

In [ ]:
archives = sorted(ARCHIVE.glob('B0_*/*/run_info.json')) + sorted(BASELINE_ROOT.glob('*/*/run_info.json'))
for run_info in archives:
    score_archive(run_info.parent)


## Bước 8 — Đồ thị RD

Mỗi đường là một phương pháp; trục x là bpp tính trên pixel ảnh gốc (log). Hàng trên: chất lượng ảnh;
hàng dưới: MegaDetector trên ảnh decode so với ảnh gốc (mAP, tỉ lệ mất con vật, tỉ lệ con vật "ảo").

In [ ]:
import matplotlib.pyplot as plt

points = {}
for line in RESULTS_REGISTRY.read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    if not str(row['exp_id']).startswith('p2dev_'):
        continue
    if row['illumination'] != 'all' or row.get('subset', 'all') != 'all':
        continue
    points.setdefault(row['exp_id'], {})[row['metric']] = row['value']

curves = {}
for exp_id, values in points.items():
    curve = exp_id[len('p2dev_'):].split('__')[0]
    curves.setdefault(curve, []).append(values)

panels = ['psnr', 'ms_ssim', 'lpips', 'map', 'missed_animal_rate', 'hallucination_rate']
figure, axes = plt.subplots(2, 3, figsize=(16, 9))
for axis, metric in zip(axes.flat, panels):
    for curve, values in sorted(curves.items()):
        series = sorted((v['bpp'], v[metric]) for v in values if 'bpp' in v and v.get(metric) is not None)
        if series:
            axis.plot(*zip(*series), marker='o', label=curve)
    axis.set_xscale('log')
    axis.set_xlabel('bpp (pixel ảnh gốc)')
    axis.set_title(metric)
    axis.grid(alpha=0.3)
axes.flat[0].legend(fontsize=8)
figure.tight_layout()
figure.savefig(P2 / 'rd_dev.png', dpi=120)
plt.show()
print('Đồ thị:', P2 / 'rd_dev.png')


## Bước 9 — Bảng kết quả để gửi lại

Mỗi dòng là một điểm (một phương pháp ở một mức nén). Chụp lại output cell này cùng `phase2/rd_dev.png`.

In [ ]:
columns = ['bpp', 'psnr', 'ms_ssim', 'lpips', 'map', 'ap_small', 'missed_animal_rate',
           'hallucination_rate', 'empty_fp_rate']
print(f"{'điểm':52s}" + ''.join(f'{c[:10]:>11s}' for c in columns))
for exp_id, values in sorted(points.items()):
    print(f'{exp_id[len("p2dev_"):]:52s}' + ''.join(
        f'{values[c]:11.4f}' if values.get(c) is not None else f"{'-':>11s}" for c in columns))


## Bước 10 — Đẩy dữ liệu lên Drive và tự ngắt runtime

Đặt `AUTO_DISCONNECT = False` nếu muốn giữ runtime sau khi chạy xong.

In [ ]:
AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    from google.colab import drive, runtime

    drive.flush_and_unmount()  # make sure every result has reached Drive
    print('Đã đẩy dữ liệu lên Drive. Ngắt runtime.')
    runtime.unassign()
else:
    print('Xong. Nhớ ngắt runtime (Runtime → Disconnect and delete runtime).')
